# 04 词共现网络

In [22]:
import pandas as pd
import ast
import networkx as nx
from collections import Counter
import itertools

## 3.1 统计词频，过滤低频词（只保留出现次数 >= min_freq 的词）

In [23]:
# 加载之前保存的帖子表（包含 tokens 列）
posts = pd.read_excel('Weibo_Posts_with_topic.xlsx')
print(f"加载帖子数：{len(posts)}")

# 将 tokens 列从字符串转换为列表
def parse_tokens(s):
    if pd.isna(s):
        return []
    if isinstance(s, list):
        return s
    try:
        return ast.literal_eval(s)
    except:
        return [x.strip() for x in str(s).replace('[', '').replace(']', '').split(',') if x.strip()]

posts['tokens'] = posts['tokens'].apply(parse_tokens)
posts = posts[posts['tokens'].map(len) > 0].copy()
print(f"有效帖子数：{len(posts)}")

texts = posts['tokens'].tolist()
print(f"文档数：{len(texts)}")

加载帖子数：893
有效帖子数：893
文档数：893


In [36]:
# 可调整参数
min_freq = 5     # 只保留出现次数 ≥ min_freq 的词
min_cooccur = 3   # 只保留共现次数 ≥ min_cooccur 的边

# 统计词频
word_freq = Counter()
for tokens in texts:
    word_freq.update(set(tokens))   # 同一文档内去重
valid_words = {w for w, cnt in word_freq.items() if cnt >= min_freq}
print(f"有效词数：{len(valid_words)}")

有效词数：610


## 3.2 统计词对共现次数

In [37]:
cooccur = Counter()
for tokens in texts:
    words_in_doc = [w for w in set(tokens) if w in valid_words]
    if len(words_in_doc) < 2:
        continue
    for w1, w2 in itertools.combinations(sorted(words_in_doc), 2):
        cooccur[(w1, w2)] += 1

## 3.3 构建图并导出边列表

In [38]:
G = nx.Graph()
for (w1, w2), weight in cooccur.items():
    if weight >= min_cooccur:
        G.add_edge(w1, w2, weight=weight)

print(f"节点数：{G.number_of_nodes()}, 边数：{G.number_of_edges()}")
nx.write_edgelist(G, 'word_cooccurrence.txt', data=False)
print("已导出边列表：word_cooccurrence.txt")



节点数：588, 边数：7506
已导出边列表：word_cooccurrence.txt
